# 18 — Debugging nonlinear solvers systematically

**Objectives.** Separate formulation, initialization, evaluation, and convergence failures; read return statuses; inspect infeasibilities and KKT residuals.

A useful order is: (1) confirm shapes/units and finite values, (2) evaluate every residual at the initial guess, (3) inspect variable and constraint bounds, (4) scale variables/residuals/objective, (5) verify derivatives, (6) inspect sparsity and rank, (7) read solver statistics, then (8) change solver options.


In [ ]:
import casadi as ca
from casadi_tutorial.diagnostics import solver_summary
opti=ca.Opti(); x=opti.variable(); opti.minimize((x-2)**2)
opti.subject_to(x>=3); opti.subject_to(x<=1)  # intentionally inconsistent
opti.set_initial(x,2); opti.solver('ipopt',{'print_time':False},{'print_level':0,'max_iter':20})
try:
    opti.solve()
except RuntimeError as error:
    print('Expected failure:',str(error).splitlines()[-1])
    print(solver_summary(opti.stats()))
    opti.debug.show_infeasibilities(1e-8)


In [ ]:
z=ca.MX.sym('z'); dangerous=ca.Function('dangerous',[z],[ca.sqrt(z-1)],{'regularity_check':True})
try:
    dangerous(0)
except RuntimeError as error:
    print('Expected domain failure captured:',type(error).__name__)


## Status-driven actions

- **Infeasible/restoration failed:** inspect bounds and residual blocks; consider explicit, penalized slack only when violation has physical meaning.
- **Maximum iterations:** check progress, scaling, derivatives, and initialization before increasing the limit.
- **Invalid number:** isolate the reported output row, evaluate dependencies, and protect model domains physically.
- **Search direction/singular system:** inspect redundant constraints, Jacobian rank, curvature, and nonsmooth expressions.
- **Acceptable rather than desired tolerance:** calculate application-level residuals before accepting it.

`Opti.debug.x_describe(i)` and `g_describe(i)` map vector indices back to source. IPOPT's derivative checker, limited-memory Hessian, bound relaxation, and verbosity are diagnostic tools—not permanent patches for a broken model. **Recap:** preserve the smallest failing case and report versions, status, statistics, bounds, initial values, and residuals.
